# FIN-02 | Notebook 1 — Data Loading, Joining & EDA

## 1. Setup & Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.data_loader import load_all_tables, get_table_summary


## 2. Load All Tables

In [ ]:
data_dir = '../data'
tables = load_all_tables(data_dir)
summary = get_table_summary(tables)
display(summary)


## 3. Table Previews

In [ ]:
for name, df in tables.items():
    print(f"\n--- {name.upper()} ---")
    print(f"Shape: {df.shape}")
    display(df.head())


## 4. Date Range Analysis

In [ ]:
trans = tables.get('trans', pd.DataFrame())
account = tables.get('account', pd.DataFrame())

if not trans.empty:
    print("Transaction Date Range:", trans['date'].min(), "to", trans['date'].max())
if not account.empty:
    print("Account Open Date Range:", account['date'].min(), "to", account['date'].max())


## 5. Transaction EDA

In [ ]:
if not trans.empty:
    plt.figure(figsize=(15, 10))
    
    plt.subplot(2, 2, 1)
    trans_counts = trans.groupby('account_id').size()
    sns.histplot(trans_counts, bins=50, kde=True)
    plt.title('Transaction Count per Account')
    
    plt.subplot(2, 2, 2)
    sns.countplot(data=trans, x='type')
    plt.title('Transaction Type Distribution')
    
    plt.subplot(2, 2, 3)
    trans['month'] = pd.to_datetime(trans['date']).dt.to_period('M')
    monthly_vol = trans.groupby('month').size()
    monthly_vol.plot(kind='line')
    plt.title('Monthly Transaction Volume')
    plt.xlabel('Month')
    
    plt.subplot(2, 2, 4)
    sns.histplot(trans['balance'], bins=50, kde=True)
    plt.title('Balance Distribution')
    
    plt.tight_layout()
    plt.show()


## 6. Account EDA

In [ ]:
if not account.empty:
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    sns.histplot(pd.to_datetime(account['date']), bins=50)
    plt.title('Account Open Date Distribution')
    plt.xticks(rotation=45)
    
    plt.subplot(1, 3, 2)
    sns.countplot(data=account, x='frequency')
    plt.title('Statement Frequency Breakdown')
    plt.xticks(rotation=45)
    
    plt.subplot(1, 3, 3)
    top_districts = account['district_id'].value_counts().head(10)
    sns.barplot(x=top_districts.index, y=top_districts.values, order=top_districts.index)
    plt.title('Top 10 Districts')
    
    plt.tight_layout()
    plt.show()


## 7. Churn Label Preview

In [ ]:
try:
    from src.features import OBS_END, LABEL_START, LABEL_END
    print("Churn window:", LABEL_START, "to", LABEL_END)
    # Dummy label preview if logic existed here (often done in features step)
    print("Note: Churn definition implemented in feature engineering.")
except ImportError:
    print("Could not load churn definition constants.")


## 8. Key Findings

- The data consists of multiple related tables describing accounts, transactions, and clients.
- Transaction volumes appear consistent across the observation window with some clear cyclical patterns.
- Account issuance grew steadily.
- We have enough history to construct a robust observation window before the labeling window.